In [ ]:
#imports
import pandas as pd
import numpy as np
import torch
import os
import plotly as px
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# This forces Plotly to draw the graph in a self-contained box
pio.renderers.default = 'iframe' 

# (Paste your Plotly fig code here and run fig.show() again)

# Set Keras to use PyTorch
os.environ["KERAS_BACKEND"] = "torch"
import keras

sns.set_theme(style="whitegrid")


df = pd.read_csv('gcs/Master_Data_Sheet.csv')

df.head()

df.columns = df.columns.str.replace(r'\s+', ' ', regex=True)

NUM_FEATURES = [
    'Age', 'ER (%)', 'PR (%)', 'Year of Diagnosis', 
    'Number of Tumor Core Biopsies', 'cT (Size, mm)'
]

BINARY_FEATURES = [
    'ER (0=neg; 1=pos)', 'PR (0=neg; 1=pos)', 'HER2 (0=neg; 1=pos)', 
    'Stain Normalization', 'cN (0=neg; 1=pos)', 'NAC Response (0=complete; 1=partial)'
]

TEXT_FEATURES = [
    'Adjuvant Radiation', 'Adjuvant Systemic Therapy', 'Survival Information', 
    'Image Resolution', 'Neoadjuvant Chemotherapy (NAC)'
]

CAT_FEATURES = [
    'Type', 'Magnification', 'Digital Pathology System', 
    'Histology (0=IDC; 1=ILC; 2=IMC)', 'Surgery Type', 'LVI'
]

ORDINAL_FEATURES = [
    'Grade Nottingham', 'cT (TNM, ajcc V8)', 'cN (TNM, ajcc V8)', 'RCBI'
]

ID_FEATURES = [
    'NAC', 'Acc', 'n'
]

df.nunique()

df.describe()

for col in CAT_FEATURES:
    print(f"{col} categories: {df[col].unique()}")

for col in ORDINAL_FEATURES:
    print(f"{col} categories: {df[col].unique()}")


# Assuming RCBI looks something like RCB-0, RCB-I, etc.
# Check your specific text formats, but here is the logic:
rcbi_mapping = {
    '0': 0,
    'RCB-I': 1,
    'RCB-II': 2,
    'RCB-III': 3,
    'Not reported': -1  # Explicitly label the unknowns as -1
}

# If your column is already text, map it:
df['RCBI'] = df['RCBI'].map(rcbi_mapping)
# If it already got turned into NaNs from our earlier coerce step, fill them:
df['RCBI'] = df['RCBI'].fillna(-1)

# 1. Static Histogram of Tumor Size colored by Treatment Response
plt.figure(figsize=(10, 6))
sns.histplot(
    data=df, 
    x="cT (Size, mm)", 
    hue="NAC Response (0=complete; 1=partial)", 
    multiple="stack", # Stacks the bars on top of each other
    kde=True,         # Adds a smooth distribution curve
    palette="Set1"
)
plt.title("Distribution of Tumor Size colored by NAC Response")
plt.xlabel("Tumor Size (mm)")
plt.show()


# 2. Box Plot: Does Age impact the NAC Response?
plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df, 
    x='NAC Response (0=complete; 1=partial)',      
    y='Age ',                                      
    palette='Set2'
)
plt.title('Patient Age vs. NAC Response')
plt.xlabel('NAC Response (0=Complete, 1=Partial)')
plt.ylabel('Age')
plt.show()



# 2. Static Scatter Plot: Age, Size, ER%, and Response
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df, 
    x="Age ", 
    y="cT (Size, mm)", 
    hue="NAC Response (0=complete; 1=partial)", 
    size="ER (%)",    # Changes dot size based on ER%
    sizes=(20, 200),  # Minimum and maximum dot sizes
    alpha=0.7,        # Makes dots slightly transparent to see overlaps
    palette="coolwarm"
)
plt.title("Multivariate Scatter: Age, Size, ER%, and Response")
plt.xlabel("Age")
plt.ylabel("Tumor Size (mm)")
# Move the legend outside the plot so it doesn't cover your data
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') 
plt.tight_layout()
plt.show()

# Age does not seem to be a differentiating factor or a predictor for the response variable

missing_counts = df.isnull().sum()
print("--- Missing Values Count ---")
# Only show columns that actually have missing data
print(missing_counts[missing_counts > 0])

plt.figure(figsize=(12, 6))
# This creates a solid block where yellow lines represent missing data points
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Data Map (Yellow = Missing Value)')
plt.tight_layout()
plt.show()

# Using the exact column names from your error message earlier!
cols_to_correlate = [
    'Age ', 'ER (%)', 
    'PR (%)', 
    'cT (Size, mm)', 
    'Year of Diagnosis', 
    'Number of Tumor Core Biopsies',
    'Grade Nottingham', 
    'cT (TNM, ajcc V8)', 
    'cN (TNM, ajcc V8)', 
    'HER2 (0=neg; 1=pos)',
    'NAC Response (0=complete; 1=partial)'
]

df[cols_to_correlate] = df[cols_to_correlate].apply(pd.to_numeric, errors='coerce')

plt.figure(figsize=(10, 8))
# Calculate correlation (Spearman is best for clinical/ordinal data)
corr_matrix = df[cols_to_correlate].corr(method='spearman')

sns.heatmap(
    corr_matrix, 
    annot=True,        # Show the actual correlation numbers
    cmap='coolwarm',   # Red = Positive correlation, Blue = Negative
    fmt=".2f", 
    vmin=-1, vmax=1
)
plt.title('Spearman Correlation: Clinical Features vs Treatment Response')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.countplot(
    data=df, 
    x='HER2 (0=neg; 1=pos)', 
    hue='NAC Response (0=complete; 1=partial)', 
    palette='Set1'
)
plt.title('HER2 Status vs. NAC Treatment Response')
plt.xlabel('HER2 Status (0 = Negative, 1 = Positive)')
plt.ylabel('Patient Count')
plt.legend(title='NAC Response', labels=['0 = Complete', '1 = Partial'])
plt.show()

import matplotlib.pyplot as plt
import seaborn as sns

df.columns = df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

# 2. Define the exact lists
NUM_FEATURES = [
    'Age', 'ER (%)', 'PR (%)', 'Year of Diagnosis', 
    'Number of Tumor Core Biopsies', 'cT (Size, mm)'
]
TARGET = 'NAC Response (0=complete; 1=partial)'

# 3. Double-check that Pandas actually did its job
missing_cols = [col for col in NUM_FEATURES if col not in df.columns]
if missing_cols:
    print(f"CRITICAL ERROR: The dataframe is still hiding these columns: {missing_cols}")
    print(f"Here is what the dataframe ACTUALLY has:\n{df.columns.tolist()}")
else:
    print("Columns match perfectly! Generating plots...")
    
    # 4. Generate the plots
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 10))
    axes = axes.flatten()

    for i, col in enumerate(NUM_FEATURES):
        sns.histplot(
            data=df, 
            x=col, 
            hue=TARGET, 
            multiple="stack", 
            kde=True, 
            ax=axes[i], 
            palette="Set1"
        )
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel('') 

    plt.tight_layout()
    plt.show()

cols_for_pairplot = NUM_FEATURES + [TARGET]

# Generate the Scatterplot Matrix
sns.pairplot(
    data=df[cols_for_pairplot],
    hue=TARGET,            # Color the dots based on treatment success
    palette="Set1",
    diag_kind="kde",       # Shows smooth density curves down the middle diagonal
    plot_kws={'alpha': 0.7, 's': 40}  # Makes dots slightly transparent to see overlaps
)

# Add a title at the top
plt.suptitle('Scatterplot Matrix of Numerical Clinical Features', y=1.02, fontsize=18)
plt.show()

# Fill missing treatment data with "None"
treatment_cols = ['Adjuvant Radiation', 'Adjuvant Systemic Therapy']

for col in treatment_cols:
    df[col] = df[col].fillna('None')


from sklearn.impute import SimpleImputer

# Create an imputer that fills missing numbers with the median value of that column
num_imputer = SimpleImputer(strategy='median')

# We apply this to the NUM_FEATURES list we made earlier
NUM_FEATURES = [
    'Age', 'ER (%)', 'PR (%)', 'Year of Diagnosis', 
    'Number of Tumor Core Biopsies', 'cT (Size, mm)'
]

df[NUM_FEATURES] = num_imputer.fit_transform(df[NUM_FEATURES])

df.head()

plt.figure(figsize=(12, 6))
# This creates a solid block where yellow lines represent missing data points
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Data Map (Yellow = Missing Value)')
plt.tight_layout()
plt.show()

df.to_csv('processed_data.csv')